#### Сравним `FNAME.DAT` и `FSIZE.DAT`

In [3]:
import os
import struct

# --- НАСТРОЙКИ ---
fname_file = "Data/FNAME.DAT"
fsize_file = "Data/FSIZE.DAT"

# --- 1. Чтение имен из FNAME.DAT ---
with open(fname_file, "rb") as f:
    raw_data = f.read()

# Разделяем по нулевым байтам
entries = raw_data.split(b'\x00')[:-1]

filenames = []
for entry in entries:
    name = entry.decode('ascii').rstrip(';1')
    if name.startswith('\\'):
        name = name[1:]
    filenames.append(name)

print(f"Всего имен: {len(filenames)}")
print("=" * 80)

# --- 2. Чтение смещений и размеров из FSIZE.DAT ---
with open(fsize_file, "rb") as f:
    fsize_data = f.read()

# Количество записей в FSIZE
record_count = len(fsize_data) // 8
print(f"Всего записей в FSIZE: {record_count}")
print("=" * 80)

# --- 3. Покажем записи с 0 по 15 ---
print("\nЗаписи 0-15 (с SYSTEM.CNF и исполняемым файлом):")
print("-" * 100)

for i in range(min(16, len(filenames), record_count)):
    offset, size = struct.unpack_from("<II", fsize_data, i * 8)
    name = filenames[i] if i < len(filenames) else "НЕТ ИМЕНИ"
    
    # Отметим важные файлы
    marker = ""
    if "SYSTEM.CNF" in name:
        marker = "← СИСТЕМНЫЙ ФАЙЛ"
    elif "SLUS_003.39" in name:
        marker = "← ИСПОЛНЯЕМЫЙ ФАЙЛ"
    elif "FNAME.DAT" in name:
        marker = "← ТАБЛИЦА ИМЕН"
    elif "FSIZE.DAT" in name:
        marker = "← ТАБЛИЦА РАЗМЕРОВ"
    
    print(f"{i:4d} | {name:40s} | offset=0x{offset:08X} ({offset:8d}) | size=0x{size:08X} ({size:8d}) {marker}")

# --- 4. Проверка соответствия ---
print("\n" + "=" * 80)
if len(filenames) != record_count:
    print(f"⚠️  НЕСООТВЕТСТВИЕ: Имен: {len(filenames)}, Записей в FSIZE: {record_count}")
    
    # Проверим, может быть в начале FNAME.DAT есть заголовок?
    if len(filenames) > record_count:
        print(f"\nЛишние имена (индексы {record_count}...{record_count+4}):")
        for i in range(record_count, min(record_count+5, len(filenames))):
            print(f"  {i}: {filenames[i]}")
    else:
        print(f"\nЛишние записи в FSIZE (индексы {len(filenames)}...{len(filenames)+4}):")
        for i in range(len(filenames), min(len(filenames)+5, record_count)):
            offset, size = struct.unpack_from("<II", fsize_data, i * 8)
            print(f"  {i}: offset=0x{offset:08X}, size=0x{size:08X}")
else:
    print("✅ Количество имен и записей в FSIDE совпадает!")
    
# --- 5. Покажем общее количество ---
print(f"\n📊 Статистика:")
print(f"  Общий размер образа (по последнему смещению + размер): ???")
if record_count > 0:
    last_offset, last_size = struct.unpack_from("<II", fsize_data, (record_count-1) * 8)
    total_size = last_offset + last_size
    print(f"  Последний файл: offset=0x{last_offset:08X}, size=0x{last_size:08X}")
    print(f"  Примерный размер образа: {total_size:,} байт ({total_size / (1024*1024):.2f} MB)")

Всего имен: 391
Всего записей в FSIZE: 195

Записи 0-15 (с SYSTEM.CNF и исполняемым файлом):
----------------------------------------------------------------------------------------------------
   0 | SYSTEM.CNF                               | offset=0x00000800 (    2048) | size=0x00000800 (    2048) ← СИСТЕМНЫЙ ФАЙЛ
   1 | FSECT.DAT                                | offset=0x00002000 (    8192) | size=0x00048000 (  294912) 
   2 | FNAME.DAT                                | offset=0x00015800 (   88064) | size=0x00058800 (  362496) ← ТАБЛИЦА ИМЕН
   3 | SLUS_003.39                              | offset=0x00093000 (  602112) | size=0x00050800 (  329728) ← ИСПОЛНЯЕМЫЙ ФАЙЛ
   4 | NAME.BIN                                 | offset=0x0003C800 (  247808) | size=0x00055800 (  350208) 
   5 | ADV.BIN                                  | offset=0x0005B000 (  372736) | size=0x00000800 (    2048) 
   6 | BTLP.BIN                                 | offset=0x00002000 (    8192) | size=0x00034000 (  2129

In [4]:
import struct

# --- Чтение FSECT.DAT ---
with open("Data/FSECT.DAT", "rb") as f:
    fsect_data = f.read()

# Количество записей
record_count = len(fsect_data) // 4
print(f"Всего записей в FSECT.DAT: {record_count}")
print("=" * 80)

# --- Анализ первых 30 записей ---
print("\nПервые 30 записей (секторы/смещения):")
print("-" * 60)

for i in range(min(30, record_count)):
    value = struct.unpack_from("<I", fsect_data, i * 4)[0]
    # Если это сектора по 2048 байт, покажем также смещение в байтах
    sector_size = 2048
    byte_offset = value * sector_size
    
    print(f"{i:4d}: 0x{value:08X} ({value:8d}) -> байтовое смещение: 0x{byte_offset:08X} ({byte_offset:10d})")

# --- Проверка на паттерны ---
print("\n" + "=" * 80)
print("Поиск паттернов:")

# Проверим, начинаются ли значения с 0x17 (23) - это может быть первый сектор данных
first_value = struct.unpack_from("<I", fsect_data, 0)[0]
print(f"Первое значение: {first_value} (0x{first_value:08X})")

# Проверим, монотонно ли возрастают значения
print("\nПроверка монотонности (первые 20):")
is_monotonic = True
for i in range(1, min(20, record_count)):
    prev = struct.unpack_from("<I", fsect_data, (i-1) * 4)[0]
    curr = struct.unpack_from("<I", fsect_data, i * 4)[0]
    if curr <= prev:
        print(f"  Нарушение на индексе {i}: {prev} -> {curr}")
        is_monotonic = False
        break

if is_monotonic:
    print("  ✅ Значения монотонно возрастают (это хорошо для таблицы смещений)")

# --- Проверим соответствие с FSIZE.DAT ---
with open("Data/FSIZE.DAT", "rb") as f:
    fsize_data = f.read()

fsize_count = len(fsize_data) // 8
print(f"\nКоличество записей в FSIZE.DAT: {fsize_count}")

if fsize_count == record_count:
    print("✅ Количество записей в FSECT и FSIZE совпадает!")
    
    # Сравним первые 10
    print("\nСравнение первых 10 записей:")
    print("Индекс | FSECT (сектор) | FSIZE (offset, size)")
    print("-" * 70)
    
    for i in range(min(10, record_count)):
        fsect_val = struct.unpack_from("<I", fsect_data, i * 4)[0]
        fsize_offset, fsize_size = struct.unpack_from("<II", fsize_data, i * 8)
        
        # Если FSECT хранит сектора, то byte_offset = fsect_val * 2048
        byte_offset_from_sect = fsect_val * 2048
        
        print(f"{i:5d} | {fsect_val:8d} (0x{fsect_val:08X}) | offset={fsize_offset:8d} (0x{fsize_offset:08X}), size={fsize_size:8d}")
        
        # Проверяем соответствие
        if byte_offset_from_sect == fsize_offset:
            print(f"       ✅ Совпадает с FSIZE offset!")
        else:
            diff = byte_offset_from_sect - fsize_offset
            print(f"       ⚠️  Расхождение: {byte_offset_from_sect} vs {fsize_offset} (разница {diff})")
else:
    print(f"⚠️  Количество записей не совпадает: FSECT={record_count}, FSIZE={fsize_count}")

# --- Покажем последние 10 записей ---
print("\n" + "=" * 80)
print("Последние 10 записей:")
print("-" * 60)

for i in range(max(0, record_count-10), record_count):
    value = struct.unpack_from("<I", fsect_data, i * 4)[0]
    byte_offset = value * 2048
    print(f"{i:4d}: 0x{value:08X} ({value:8d}) -> байтовое смещение: 0x{byte_offset:08X} ({byte_offset:10d})")

Всего записей в FSECT.DAT: 391

Первые 30 записей (секторы/смещения):
------------------------------------------------------------
   0: 0x00000017 (      23) -> байтовое смещение: 0x0000B800 (     47104)
   1: 0x00000018 (      24) -> байтовое смещение: 0x0000C000 (     49152)
   2: 0x00000019 (      25) -> байтовое смещение: 0x0000C800 (     51200)
   3: 0x0000001D (      29) -> байтовое смещение: 0x0000E800 (     59392)
   4: 0x000000AD (     173) -> байтовое смещение: 0x00056800 (    354304)
   5: 0x000000D8 (     216) -> байтовое смещение: 0x0006C000 (    442368)
   6: 0x00000189 (     393) -> байтовое смещение: 0x000C4800 (    804864)
   7: 0x000002AF (     687) -> байтовое смещение: 0x00157800 (   1406976)
   8: 0x00000350 (     848) -> байтовое смещение: 0x001A8000 (   1736704)
   9: 0x000003C9 (     969) -> байтовое смещение: 0x001E4800 (   1984512)
  10: 0x00000474 (    1140) -> байтовое смещение: 0x0023A000 (   2334720)
  11: 0x0000052A (    1322) -> байтовое смещение: 0x002

In [5]:
import struct

# --- Чтение всех файлов ---
with open("Data/FNAME.DAT", "rb") as f:
    raw_names = f.read()
    # Разделяем по нулевым байтам, игнорируя последний пустой
    name_entries = raw_names.split(b'\x00')[:-1]
    filenames = [entry.decode('ascii').rstrip(';1').lstrip('\\') for entry in name_entries]

with open("Data/FSECT.DAT", "rb") as f:
    fsect_data = f.read()
    fsect_values = [struct.unpack_from("<I", fsect_data, i*4)[0] for i in range(len(fsect_data)//4)]

with open("Data/FSIZE.DAT", "rb") as f:
    fsize_data = f.read()
    fsize_pairs = [struct.unpack_from("<II", fsize_data, i*8) for i in range(len(fsize_data)//8)]

print(f"FNAME.DAT: {len(filenames)} записей")
print(f"FSECT.DAT: {len(fsect_values)} записей")
print(f"FSIZE.DAT: {len(fsize_pairs)} записей (пар offset/size)")
print("=" * 80)

# --- Анализ первых 30 файлов ---
print("\nПервые 30 файлов с их секторами и размерами:")
print("-" * 120)
print(f"{'Индекс':<6} {'Имя файла':<35} {'Сектор':<10} {'Байт offset':<15} {'Размер (FSIZE)':<15} {'Примечание'}")
print("-" * 120)

for i in range(min(30, len(filenames))):
    name = filenames[i]
    sector = fsect_values[i]
    byte_offset = sector * 2048
    
    size_info = ""
    size_value = 0
    
    if i < len(fsize_pairs):
        offset, size = fsize_pairs[i]
        size_value = size
        if offset == byte_offset:
            size_info = f"0x{size:08X} ({size})"
        else:
            size_info = f"0x{size:08X} ({size}) [offset не совпадает!]"
    else:
        size_info = "НЕТ В FSIZE"
    
    # Определяем тип файла по расширению
    file_type = ""
    if name.endswith(".CNF"):
        file_type = "← SYSTEM"
    elif name.endswith(".39"):
        file_type = "← EXECUTABLE"
    elif name.endswith(".DAT"):
        file_type = "← TABLE"
    elif name.endswith(".BIN"):
        file_type = "← DATA"
    elif name.endswith(".VB"):
        file_type = "← VAG AUDIO"
    elif name.endswith(".STR"):
        file_type = "← VIDEO"
    elif name.endswith(".EXE"):
        file_type = "← EXE"
    
    print(f"{i:<6} {name:<35} {sector:<10} 0x{byte_offset:08X} ({byte_offset:>8}) {size_info:<20} {file_type}")

# --- Анализ файлов, у которых нет размера в FSIZE ---
print("\n" + "=" * 80)
print(f"Файлы без размера в FSIZE (индексы {len(fsize_pairs)}...{len(filenames)-1}):")
print("-" * 80)

for i in range(len(fsize_pairs), min(len(filenames), len(fsize_pairs)+20)):
    name = filenames[i]
    sector = fsect_values[i]
    byte_offset = sector * 2048
    
    # Вычисляем примерный размер как разницу между текущим и следующим сектором
    if i + 1 < len(fsect_values):
        next_sector = fsect_values[i + 1]
        approx_size = (next_sector - sector) * 2048
        size_info = f"≈ {approx_size} байт (по разнице секторов)"
    else:
        size_info = "последний файл"
    
    print(f"{i:<6} {name:<50} сектор={sector:<8} 0x{byte_offset:08X} | {size_info}")

# --- Проверка: совпадают ли offset в FSIZE с FSECT? ---
print("\n" + "=" * 80)
print("Проверка соответствия FSIZE offset и FSECT (первые 10):")
print("-" * 80)

for i in range(min(10, len(fsize_pairs))):
    offset, size = fsize_pairs[i]
    sector = fsect_values[i]
    byte_offset_from_sect = sector * 2048
    
    match = "✅" if offset == byte_offset_from_sect else "❌"
    print(f"{i}: FSIZE offset={offset:8d} (0x{offset:08X}) | FSECT offset={byte_offset_from_sect:8d} (0x{byte_offset_from_sect:08X}) {match}")

# --- Вывод ---
print("\n" + "=" * 80)
print("ВЫВОДЫ:")
print(f"1. FSECT.DAT содержит номера секторов (по {2048} байт) для ВСЕХ {len(fsect_values)} файлов")
print(f"2. FSIZE.DAT содержит точные размеры ТОЛЬКО для первых {len(fsize_pairs)} файлов")
print(f"3. Для остальных {len(filenames) - len(fsize_pairs)} файлов размер можно вычислить как разницу между соседними секторами в FSECT")
print(f"4. Первые {len(fsize_pairs)} файлов имеют смещения, совпадающие с FSECT (проверено)")

FNAME.DAT: 391 записей
FSECT.DAT: 391 записей
FSIZE.DAT: 195 записей (пар offset/size)

Первые 30 файлов с их секторами и размерами:
------------------------------------------------------------------------------------------------------------------------
Индекс Имя файла                           Сектор     Байт offset     Размер (FSIZE)  Примечание
------------------------------------------------------------------------------------------------------------------------
0      SYSTEM.CNF                          23         0x0000B800 (   47104) 0x00000800 (2048) [offset не совпадает!] ← SYSTEM
1      FSECT.DAT                           24         0x0000C000 (   49152) 0x00048000 (294912) [offset не совпадает!] ← TABLE
2      FNAME.DAT                           25         0x0000C800 (   51200) 0x00058800 (362496) [offset не совпадает!] ← TABLE
3      SLUS_003.39                         29         0x0000E800 (   59392) 0x00050800 (329728) [offset не совпадает!] ← EXECUTABLE
4      NAME.BIN 